# X1 — Persona LoRA experiment (Kaggle, free T4)

Trains one LoRA adapter for **one persona** and generates answers for its held-out questions in two conditions:

- `lora` — Typhoon 2.5 4B + adapter, short system prompt, no examples
- `typhoon_prompt_only` — same base model, adapter disabled, full runtime prompt with examples

The third condition (`frontier_prompt_only`) comes from the export file. Judging happens back in the app repo
with `scripts/judge_experiment.py`, so no API keys are needed here.

**Kaggle settings:** Accelerator = GPU T4 (x2 is fine; only one GPU is used) · Internet = On ·
add your **private** dataset containing `persona_*.jsonl`.

**Privacy:** the dataset and the adapter contain what was learned from private examples.
Keep the notebook, dataset, and adapter private. Delete the dataset when the experiment ends.

Spec: `docs/LORA_EXPERIMENT.md`

In [ ]:
%%capture
!pip install -q unsloth
# If installation fails, check the current Kaggle install instructions in Unsloth's docs.

## 1. Config — edit `PERSONA_FILE` for each run

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use a single GPU

import json, random, re, shutil

PERSONA_FILE = "/kaggle/input/persona-lora-data/persona_p_123.jsonl"   # <-- change per run
BASE_MODEL   = "typhoon-ai/typhoon2.5-qwen3-4b"

MAX_SEQ_LENGTH = 2048
LORA_RANK      = 16        # lower to 8 if the adapter copies training answers
EPOCHS         = 3         # lower to 1–2 if the adapter copies training answers
LEARNING_RATE  = 2e-4
BATCH_SIZE     = 2
GRAD_ACCUM     = 4         # effective batch size 8
SEED           = 3407

# Sampling settings suggested on the Typhoon 2.5 model card
GEN_KWARGS = dict(max_new_tokens=512, do_sample=True, temperature=0.6, top_p=0.95, repetition_penalty=1.05)

OUT_DIR = "/kaggle/working"
random.seed(SEED)

## 2. Load and validate the export file

In [ ]:
meta, train, heldout = None, [], []
with open(PERSONA_FILE, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        if row["type"] == "meta":
            meta = row
        elif row["split"] == "train":
            train.append(row)
        elif row["split"] == "heldout":
            heldout.append(row)

assert meta is not None, "missing meta line"
assert len(heldout) >= 5, "need at least 5 held-out examples"
assert all(r["source"] == "real" for r in heldout), "held-out examples must be real"

heldout_questions = {r["question"].strip() for r in heldout}
assert not any(r["question"].strip() in heldout_questions for r in train), "held-out question leaked into train"

PERSONA_ID = meta["persona_id"]
n_real = sum(r["source"] == "real" for r in train)
print(f"Persona {PERSONA_ID} ({meta['display_name']}): "
      f"{n_real} real + {len(train) - n_real} synthetic train, {len(heldout)} held-out")

## 3. Load the base model (4-bit) and add LoRA

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## 4. Build the training set

Each example becomes: short system prompt → question → answer. The same short system prompt is used at inference,
so the adapter (not the prompt) has to carry the style. Check the printed sample: the chat template must look right
before training.

In [ ]:
from datasets import Dataset

def to_text(row):
    messages = [
        {"role": "system", "content": meta["lora_system_prompt"]},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["answer"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)

random.shuffle(train)
train_ds = Dataset.from_list([{"text": to_text(r)} for r in train])
print(train_ds[0]["text"][:1500])

## 5. Train (loss only on the assistant answers)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir=f"{OUT_DIR}/checkpoints",
        save_strategy="no",
        report_to="none",
    ),
)

# Qwen3-style chat markers (Typhoon 2.5 is Qwen3-based). If the printed sample above uses
# different markers, change these two strings to match.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print(stats)

## 6. Generate held-out answers in both conditions

In [ ]:
FastLanguageModel.for_inference(model)
THINK_RE = re.compile(r"<think>.*?</think>", flags=re.S)

def generate(system_prompt, question):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, **GEN_KWARGS)
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return THINK_RE.sub("", text).strip()

results = []
for r in heldout:
    results.append({"example_id": r["id"], "condition": "lora",
                    "answer": generate(meta["lora_system_prompt"], r["question"])})

with model.disable_adapter():   # same base model, adapter off
    for r in heldout:
        results.append({"example_id": r["id"], "condition": "typhoon_prompt_only",
                        "answer": generate(meta["runtime_system_prompt"], r["question"])})

print(f"Generated {len(results)} answers")

## 7. Quick look (the real judging happens in the app repo)

In [ ]:
by_id = {}
for res in results:
    by_id.setdefault(res["example_id"], {})[res["condition"]] = res["answer"]

for r in heldout[:3]:
    print("=" * 80)
    print("Q:", r["question"])
    print("\n--- REAL ---\n", r["answer"])
    print("\n--- LORA ---\n", by_id[r["id"]]["lora"])
    print("\n--- TYPHOON PROMPT-ONLY ---\n", by_id[r["id"]]["typhoon_prompt_only"])
    print("\n--- FRONTIER PROMPT-ONLY ---\n", r.get("baseline_answer", "(missing)"))

## 8. Memorization check

Character 8-grams (works for Thai, which has no spaces). For each `lora` answer, find the real training answer it
overlaps with most. Overlap above 0.35 means the adapter may be copying private text.

In [ ]:
def char_ngrams(text, n=8):
    s = " ".join(text.lower().split())
    return {s[i:i + n] for i in range(max(0, len(s) - n + 1))}

real_train = [(r["id"], char_ngrams(r["answer"])) for r in train if r["source"] == "real"]
flags = []
for res in results:
    if res["condition"] != "lora":
        continue
    grams = char_ngrams(res["answer"])
    if not grams:
        continue
    best_id, best = max(((tid, len(grams & tg) / len(grams)) for tid, tg in real_train),
                        key=lambda x: x[1], default=(None, 0.0))
    res["max_train_overlap"] = round(best, 3)
    if best > 0.35:
        flags.append((res["example_id"], best_id, round(best, 3)))

print("Possible memorization:", flags if flags else "none")

## 9. Save results and adapter

In [ ]:
results_path = f"{OUT_DIR}/results_{PERSONA_ID}.jsonl"
with open(results_path, "w", encoding="utf-8") as f:
    for res in results:
        f.write(json.dumps(res, ensure_ascii=False) + "\n")

adapter_dir = f"{OUT_DIR}/adapter_{PERSONA_ID}"
model.save_pretrained(adapter_dir)          # saves only the LoRA adapter (small)
tokenizer.save_pretrained(adapter_dir)
shutil.make_archive(adapter_dir, "zip", adapter_dir)

print("Download from the Output panel:")
print(" -", results_path)
print(" -", adapter_dir + ".zip   (private — do not upload publicly)")

## Next

1. Download `results_<persona_id>.jsonl`.
2. In the app repo: `scripts/judge_experiment.py --export persona_<id>.jsonl --results results_<id>.jsonl`
3. Restart the session (Run → Restart) before the next persona, change `PERSONA_FILE`, and run all again.